Checking different VISA connections

In [8]:
import pyvisa

rm = pyvisa.ResourceManager()
resources = rm.list_resources()
for res in resources:
    try:
        inst = rm.open_resource(res)
        idn = inst.query('*IDN?')
        print(f"{res} → {idn}")
    except Exception:
        print(f"{res} → no response")

TCPIP0::A-34461A-00000::hislip0::INSTR → no response
TCPIP0::A-34461A-00000::inst0::INSTR → no response
TCPIP0::A-34461A-0000-2.local::hislip0::INSTR → no response
TCPIP0::A-34461A-0000-2.local::inst0::INSTR → Keysight Technologies,34461A,MY60040662,A.03.03-03.15-03.03-00.52-04-03

TCPIP0::10.93.130.7::hislip0::INSTR → Rohde&Schwarz,RTO6,1802.0001k04/101518,5.50.2.0

TCPIP0::10.93.130.7::inst0::INSTR → Rohde&Schwarz,RTO6,1802.0001k04/101518,5.50.2.0

ASRL1::INSTR → *IDN?E01

ASRL3::INSTR → no response
GPIB0::5::INSTR → KEPCO,BOP1KW 36-28 03-24-2011 ,A104417, 3.35

GPIB0::8::INSTR → no response


In [ ]:
# Checking connection with SR850 LIA and KEPCO PS
import pyvisa
rm = pyvisa.ResourceManager()

SR_ADDR = 'GPIB0::8::INSTR'  # replace with correct address
inst = rm.open_resource(SR_ADDR)
inst.timeout = 10000
inst.write_termination = '\n'
inst.read_termination = '\n'

# identity query
try:
    idn = inst.query('*IDN?')
    print("Instrument ID:", idn)
except pyvisa.errors.VisaIOError as e:
    print("GPIB read error:", e)

SR_ADDR = 'GPIB0::5::INSTR'  # replace with correct address
inst = rm.open_resource(SR_ADDR)
inst.timeout = 10000
inst.write_termination = '\n'
inst.read_termination = '\n'

# identity query
try:
    idn = inst.query('*IDN?')
    print("Instrument ID:", idn)
except pyvisa.errors.VisaIOError as e:
    print("GPIB read error:", e)



Instrument ID: Stanford_Research_Systems,SR850,s/n26070,ver1.00 
Instrument ID: KEPCO,BOP1KW 36-28 03-24-2011 ,A104417, 3.35


Checking the SR850 connection

In [ ]:
from pymeasure.instruments import Instrument
from pymeasure.instruments.validators import strict_discrete_set

class SR850(Instrument):
    """
    Driver for the Stanford Research SR850 Lock-in Amplifier.
    Supports GPIB or serial RS-232 through a VISA resource.
    """

    def __init__(self, adapter, name="SR850 Lock-in Amplifier", **kwargs):
        super().__init__(adapter, name, **kwargs)

        # Terminations:
        self.adapter.connection.write_termination = "\n"
        self.adapter.connection.read_termination = "\n"

    # -----------------------
    # Core instrument queries
    # -----------------------

    id = Instrument.measurement(
        "*IDN?",
        """The identity query."""
    )

    frequency = Instrument.measurement(
        "FREQ?",
        """Reference frequency in Hz."""
    )

    sensitivity = Instrument.control(
        "SENS %d",
        "SENS?",
        """Sensitivity setting index (0–26).""",
        validator=strict_discrete_set,
        values=range(27)
    )

    time_constant = Instrument.control(
        "TC %d",
        "TC?",
        """Time constant index (0–19).""",
        validator=strict_discrete_set,
        values=range(20)
    )

    x = Instrument.measurement("OUTP? 1", "X output")
    y = Instrument.measurement("OUTP? 2", "Y output")
    r = Instrument.measurement("OUTP? 3", "R output")
    theta = Instrument.measurement("OUTP? 4", "Theta output")
inst.sensitivity = 10
print(inst.sensitivity)


TypeError: not all arguments converted during string formatting

Checking the KEPCO connection

In [3]:
"""
--------------------------------------
This script demonstrates safe operation of a Kepco BOP 1kW supply 
via GPIB (PyVISA), using the SCPI command set supported by the 2011 firmware.
 
Features:
- Initialize in constant current (CC) mode
- Apply a current setpoint with a voltage compliance limit
- Enable the output and monitor actual current/voltage in real time
- Shut down safely and verify that the output goes back to zero
"""
 
import time
import pyvisa
 
# -----------------------------
# User parameters
# -----------------------------
ADDR = "GPIB0::5::INSTR"   # GPIB address of the Kepco device
TARGET_CURRENT = 2       # Target current in amperes
VOLT_LIMIT    = 5.0        # Voltage compliance (maximum allowed voltage in V)
MONITOR_TIME  = 10         # How long to keep output enabled, in seconds
 
# -----------------------------
# Establish connection
# -----------------------------
rm = pyvisa.ResourceManager()
psu = rm.open_resource(ADDR)
psu.timeout = 3000
psu.write_termination = '\n'
psu.read_termination  = '\n'
 
# -----------------------------
# Identify device and check status
# -----------------------------
print("Device ID:", psu.query("*IDN?").strip())
outp = psu.query("OUTP?").strip()
print("Current output state:", outp)
 
outp = psu.query("OUTP?").strip()
print("Current output state:", outp)

# Force-OFF before proceeding
if outp == "1":
    print("Output was ON — turning it OFF for safety.")
    psu.write("OUTP OFF")
    time.sleep(0.5)
 
# -----------------------------
# Reset and configure source
# -----------------------------
psu.write("*RST")                        # Reset to default state
psu.write("*CLS")                        # Clear status/error registers
psu.write("FUNC CURR")                   # Select constant current (CC) mode
psu.write(f"CURR {TARGET_CURRENT}")      # Set current setpoint
psu.write(f"VOLT {VOLT_LIMIT}")          # Set voltage compliance
psu.write("OUTP ON")                     # Enable output
 
print(f"\nOutput enabled at {TARGET_CURRENT} A (compliance {VOLT_LIMIT} V).")
print(f"Holding for {MONITOR_TIME} seconds...\n")
 
# -----------------------------
# Print setpoints
# -----------------------------
print("Configured setpoints:")
print("  Current setpoint (CURR?):", psu.query("CURR?").strip(), "A")
print("  Voltage limit (VOLT?):   ", psu.query("VOLT?").strip(), "V")
 
# -----------------------------
# Monitor actual output
# -----------------------------
for i in range(MONITOR_TIME):
    time.sleep(1)
    measI = psu.query("MEAS:CURR?").strip()
    measV = psu.query("MEAS:VOLT?").strip()
    print(f"{i+1:2d}s: Actual I = {measI} A, Actual V = {measV} V")
 
# -----------------------------
# Disable output and verify
# -----------------------------
psu.write("OUTP OFF")
print("\nOutput disabled.")
 
time.sleep(1)  # Give some time for current to decay
 
measI = psu.query("MEAS:CURR?").strip()
measV = psu.query("MEAS:VOLT?").strip()
print("After shutdown:")
print("  Actual current:", measI, "A")
print("  Actual voltage:", measV, "V")
 
# -----------------------------
# Close connection
# -----------------------------
psu.close()
print("\nConnection closed. Script finished.")

Device ID: KEPCO,BOP1KW 36-28 03-24-2011 ,A104417, 3.35
Current output state: 1
Current output state: 1
Output was ON — turning it OFF for safety.

Output enabled at 2 A (compliance 5.0 V).
Holding for 10 seconds...

Configured setpoints:
  Current setpoint (CURR?): 2 A
  Voltage limit (VOLT?):    5 V
 1s: Actual I = 1.9733 A, Actual V = 2.1319 V
 2s: Actual I = 1.9724 A, Actual V = 2.1332 V
 3s: Actual I = 1.9724 A, Actual V = 2.1319 V
 4s: Actual I = 1.9724 A, Actual V = 2.1319 V
 5s: Actual I = 1.9724 A, Actual V = 2.1319 V
 6s: Actual I = 1.9743 A, Actual V = 2.1332 V
 7s: Actual I = 1.9743 A, Actual V = 2.1332 V
 8s: Actual I = 1.9724 A, Actual V = 2.1332 V
 9s: Actual I = 1.9724 A, Actual V = 2.1319 V
10s: Actual I = 1.9724 A, Actual V = 2.1319 V

Output disabled.
After shutdown:
  Actual current: -0.0096 A
  Actual voltage: -0.0040 V

Connection closed. Script finished.


In [12]:
import time
import pyvisa


class KepcoPS:
    def __init__(self, address="GPIB0::5::INSTR"):
        rm = pyvisa.ResourceManager()
        self.inst = rm.open_resource(address)
        self.inst.timeout = 3000
        self.inst.write_termination = '\n'
        self.inst.read_termination = '\n'

        print("Connected to:", self.idn)

        # Safety: turn OFF output before enabling
        if self.output_state:
            print("Output was ON — turning OFF for safety.")
            self.output_state = False

        self.reset()
        self.clear()


    def sleep_with_countdown(self, seconds):
        for i in range(seconds):
            print(f"Waiting... {seconds - i} s remaining", end="\r")
            time.sleep(1)
        print(" " * 40, end="\r")
        
    # ---------------------------------------------------------------
    # Convenience Properties
    # ---------------------------------------------------------------

    @property
    def idn(self):
        """Return identification string."""
        return self.inst.query("*IDN?").strip()

    @property
    def output_state(self):
        """Return True if output is ON."""
        return self.inst.query("OUTP?").strip() == "1"

    @output_state.setter
    def output_state(self, value: bool):
        """Enable or disable output."""
        cmd = "ON" if value else "OFF"
        self.inst.write(f"OUTP {cmd}")
        time.sleep(0.2)

    # ---------------------------------------------------------------
    # Reset and Clear
    # ---------------------------------------------------------------

    def reset(self):
        self.inst.write("*RST")

    def clear(self):
        self.inst.write("*CLS")

    # ---------------------------------------------------------------
    # Current setpoint (A) — CONSTANT CURRENT MODE
    # ---------------------------------------------------------------

    @property
    def current(self):
        """Return the *measured* output current."""
        return float(self.inst.query("MEAS:CURR?"))

    @current.setter
    def current(self, value):
        """
        Set the *target* current.
        The device stays in CC mode.
        """
        self.inst.write("FUNC CURR")  # make sure CC mode is active
        self.inst.write(f"CURR {value}")
        print(f"Set current to {value} A")
        self.sleep_with_countdown(5)

    # ---------------------------------------------------------------
    # Voltage Compliance (V)
    # ---------------------------------------------------------------

    @property
    def voltage_limit(self):
        """Voltage compliance limit."""
        return float(self.inst.query("VOLT?"))

    @voltage_limit.setter
    def voltage_limit(self, value):
        self.inst.write(f"VOLT {value}")
        print(f"Set voltage limit to {value} V")
        time.sleep(0.1)

    # ---------------------------------------------------------------
    # Measurement
    # ---------------------------------------------------------------

    @property
    def measured_voltage(self):
        return float(self.inst.query("MEAS:VOLT?"))

    # ---------------------------------------------------------------
    # Startup and Shutdown sequences
    # ---------------------------------------------------------------

    def startup(self):
        """
        Safe startup like your original script.
        """
        print("Initializing KEPCO in CC mode...")
        self.reset()
        self.clear()

        self.inst.write("FUNC CURR")

        print("Enabling output...")
        self.output_state = True
        self.sleep_with_countdown(10)

    def shutdown(self):
        """Safe output disable."""
        print("Shutting down…")
        self.output_state = False
        self.sleep_with_countdown(10)

        i = self.current
        v = self.measured_voltage

        print(f"After shutdown: I = {i} A, V = {v} V")

    # ---------------------------------------------------------------
    # Close session
    # ---------------------------------------------------------------

    def close(self):
        self.inst.close()
        print("KEPCO connection closed.")

#kepco = KepcoPS("GPIB0::5::INSTR")
#kepco.startup(0)
#kepco.current = 2.0          # set current
#print(kepco.current)         # read actual current
#kepco.shutdown()

**Bruker Connection** \
REM/ Check local/remote state 0: local 1: remote
REM= Set local/remote 0: local 1: remote


In [11]:
# Class Bruker Power Supply

import time
import re
import serial
import numpy as np


class BrukerPS:
    """
    PyMeasure-style instrument wrapper for the Bruker magnet power supply.
    Provides .current get/set and handles serial communication internally.
    """

    def __init__(self, port="COM1", baudrate=9600, timeout=1):
        self.ser = serial.Serial(
            port=port,
            baudrate=baudrate,
            bytesize=8,
            parity=serial.PARITY_NONE,
            stopbits=1,
            timeout=timeout
        )
        print(f"Connected to Bruker PSU on {port}")

    # ---------------------------------------------------------
    # Utility functions
    # ---------------------------------------------------------
    def sleep_with_countdown(self, seconds):
        for i in range(seconds):
            print(f"Waiting... {seconds - i} s remaining", end="\r")
            time.sleep(1)
        print(" " * 40, end="\r")

    def send_cmd(self, cmd):
        """Send a command and return the string response."""
        self.ser.write((cmd + "\r").encode())
        time.sleep(0.2)
        resp = self.ser.read(200).decode(errors="ignore").strip()
        print(f"Sent: {cmd}  ->  Response: {resp}")
        return resp

    def extract_current(self, s):
        match = re.search(r"([-+]?\d*\.?\d+)\s*A", s)
        if match:
            return float(match.group(1))
        return None

    # ---------------------------------------------------------
    # PyMeasure-style CURRENT PROPERTY
    # ---------------------------------------------------------

    @property
    def current(self):
        """Read the actual output current (A)."""
        resp = self.send_cmd("CHN/")
        value = self.extract_current(resp)
        return value

    @current.setter
    def current(self, value):
        """Set the PSU output current (A)."""
        print(f"Setting Bruker PSU current to {value} A")
        self.send_cmd(f"CUR={value}")
        self.sleep_with_countdown(15)

        # Check if the current actually settled
        cur = self.current
        while np.round(cur, 2) != np.round(value, 2):
            print("Current deviated; retrying...")
            self.startup()
            self.send_cmd(f"CUR={value}")
            cur = self.current

        print("Current applied successfully.")

    # ---------------------------------------------------------
    # Commands for the device
    # ---------------------------------------------------------

    def startup(self):
        self.send_cmd("RST=0")
        print("System status:")
        self.send_cmd("STA/")
        print("Set to remote mode:")
        self.send_cmd("REM/")
        print("Turn on power supply...")
        self.send_cmd("DCP=1")
        self.sleep_with_countdown(15)

        # Safety check
        dcp = self.send_cmd("DCP/")
        if dcp == "DCP/ 0":
            self.send_cmd("DCP=1")
            self.sleep_with_countdown(15)

        print("Bruker PSU is ready to rumble.")

    def shutdown(self):
        print("Turning off Bruker PSU...")
        self.send_cmd("DCP=0")
        self.ser.close()
        print("Serial port closed.")

    def measure(self):
        print("Current:", self.send_cmd("CHN/"))
        print("Voltage:", self.send_cmd("CHV/"))
        print("Referenced current:", self.send_cmd("CUR/"))


#psu = BrukerPS("COM1")
#psu.startup()

#psu.current = 3.0   # sets the current
#print(psu.current)  # reads the current

#psu.shutdown()


**PyMeasure**


In [34]:
# Check which devices are available
from pymeasure.adapters import VISAAdapter
import pyvisa

def list_available_devices():
    print("\n=== Scanning for VISA Instruments ===\n")

    rm = pyvisa.ResourceManager()
    resources = rm.list_resources()

    if not resources:
        print("No VISA instruments found.")
        return

    for res in resources:
        print(f"Found resource: {res}")
        
        try:
            # Try opening using PyMeasure's VISAAdapter
            adapter = VISAAdapter(res, timeout=2000)
            idn = adapter.ask("*IDN?")
            print(f"  *IDN?: {idn.strip()}")
        except Exception as e:
            print(f"  Unable to query *IDN?  ({type(e).__name__}): {e}")
        
        print()

if __name__ == "__main__":
    list_available_devices()



=== Scanning for VISA Instruments ===

Found resource: TCPIP0::A-34461A-00000::hislip0::INSTR
  Unable to query *IDN?  (VisaIOError): VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.

Found resource: TCPIP0::A-34461A-00000::inst0::INSTR
  Unable to query *IDN?  (VisaIOError): VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.

Found resource: TCPIP0::A-34461A-0000-2.local::hislip0::INSTR
  Unable to query *IDN?  (VisaIOError): VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.

Found resource: TCPIP0::A-34461A-0000-2.local::inst0::INSTR
  *IDN?: Keysight Technologies,34461A,MY60040662,A.03.03-03.15-03.03-00.52-04-03

Found resource: TCPIP0::10.93.130.7::hislip0::INSTR
  *IDN?: Rohde&Schwarz,RTO6,1802.0001k04/101518,5.50.2.0

Found r

**Working Code**

In [2]:
import numpy as np

def sleep_with_countdown(seconds):
    for i in range(seconds):
        print(f"Waiting... {seconds - i} s remaining", end="\r")
        time.sleep(1)
    print(" " * 40, end="\r")
    

def angular_scan(
    Bmag,
    instrument_x=None,
    instrument_y=None,
    angles_rad=np.linspace(0, 2*np.pi, 100),
    verbose=True
):
    '''Bmag in mT'''
    # Calibration constants (mT/A)
    k_x = 9.24633
    k_y = 3.84322

    Ix = []
    Iy = []
    Bvals = []

    for theta in angles_rad:
        print(theta)
        # Field components
        Bx = Bmag * np.cos(theta)
        By = Bmag * np.sin(theta)

        # Required currents
        current_x = Bx / k_x
        current_y = By / k_y

        Ix.append(current_x)
        Iy.append(current_y)

        # Magnitude check (due to rounding/calibration differences)
        Bvals.append(np.sqrt(Bx**2 + By**2))

        # --- Apply currents if PyMeasure instruments are provided ---
        if instrument_x is not None:
            instrument_x.startup()
            try:
                instrument_x.current = current_x     # Common PyMeasure driver pattern
                print('set current to', current_x)
            except:
                instrument_x.set_current(current_x)  # Alternative naming

        if instrument_y is not None:
            instrument_y.startup()
            try:
                instrument_y.current = current_y
                print('set current bruker to', current_y)
            except:
                instrument_y.set_current(current_y)
        sleep_with_countdown(29)
    # Combine results
    scan_data = list(zip(angles_rad, Ix, Iy, Bvals))

    # Optional printing
    if verbose:
        print(f"{'Angle (deg)':>10}   {'Ix (A)':>10}   {'Iy (A)':>10}   {'|B| (mT)':>10}")
        for angle, x, y, bmag in scan_data:
            print(f"{np.degrees(angle):10.2f}   {x:10.5f}   {y:10.5f}   {bmag:10.5f}")

    return scan_data


# Close bruker serial connection to prevent error
import serial
import serial.tools.list_ports
ports = serial.tools.list_ports.comports()
print("Active ports:", [p.device for p in ports])

       
psu1 = KepcoPS()
psu1.shutdown()
psu2 = BrukerPS()
psu2.shutdown()
angular_scan(1, instrument_x=psu1, instrument_y=psu2, verbose=True)

Active ports: ['COM3', 'COM1']


NameError: name 'KepcoPS' is not defined

In [ ]:
# waste
# calculatecurrents for list of angles or list of Bmagnitudes
if type(angle) == list or type(angle) == np.ndarray:
    thetas = np.deg2rad(angle)
    Ix = []
    Iy = []

    for theta in thetas:
        Bx = Bmag * np.cos(theta)
        By = Bmag * np.sin(theta)

        Ix.append(Bx / k_x)
        Iy.append(By / k_y)
    return Ix, Iy

if type(Bmag) == list or type(Bmag) == np.ndarray:
    theta = np.deg2rad(angle)
    Bxs = []
    Bys = []

    for B in Bmag:
        Bx = B * np.cos(theta)
        By = B * np.sin(theta)

        Bxs.append(Bx / k_x)
        Bys.append(By / k_y)
    return Bxs, Bys